In [1]:
import os

# Dossier pour les dépendances Spark
os.environ["HOME"] = "/tmp"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7 "
    "--conf spark.jars.ivy=/tmp/.ivy2 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Kafka_Weather_Streaming")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .getOrCreate()
)

print("Spark fonctionne :", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73301dfc-725c-44a2-9c52-010eacf4ce99;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 699ms :: artifacts dl 48ms
	:: modules in use:

Spark fonctionne : 3.5.7


In [2]:
try:
    df_kafka = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", "weather_stream")
        
        .option("startingOffsets", "latest")
        .load()
    )

    print("Connecteur Kafka disponible !")
    df_kafka.printSchema()

except Exception as e:
    print("Erreur Kafka :")
    print(str(e)[:1500])

Connecteur Kafka disponible !
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [3]:
from pyspark.sql.functions import (
    col, from_json, when, to_json, struct
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType
)

# Structure des données reçues de l'API Open-Meteo
schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", StructType([
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType())
    ]))
])

# Conversion des messages Kafka JSON
df_weather = (
    df_kafka
    .select(
        from_json(
            col("value").cast("string"),
            schema
        ).alias("data"),
        col("timestamp").alias("event_time")
    )
    .select("data.*", "event_time")
)

# Transformation et calcul des alertes
df_transformed = (
    df_weather
    .select(
        col("event_time"),
        col("city"),
        col("country"),
        col("current.temperature_2m").alias("temperature"),
        col("current.wind_speed_10m").alias("windspeed"),
        col("current.weather_code").alias("weather_code")
    )
    .withColumn(
        "wind_alert_level",
        when(col("windspeed") < 10, "level_0")
        .when(col("windspeed") <= 20, "level_1")
        .otherwise("level_2")
    )
    .withColumn(
        "heat_alert_level",
        when(col("temperature") < 25, "level_0")
        .when(col("temperature") <= 35, "level_1")
        .otherwise("level_2")
    )
    .filter(
        col("city").isNotNull()
        & col("temperature").isNotNull()
        & col("windspeed").isNotNull()
    )
)

print("Transformation des données configurée !")
df_transformed.printSchema()

Transformation des données configurée !
root
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- wind_alert_level: string (nullable = false)
 |-- heat_alert_level: string (nullable = false)



In [1]:
import socket

try:
    ip = socket.gethostbyname("kafka")
    print("Adresse IP Kafka :", ip)

    connexion = socket.create_connection(
        ("kafka", 9092),
        timeout=5
    )

    print("Connexion Kafka réussie !")
    connexion.close()

except Exception as e:
    print("Erreur Kafka :", e)

Adresse IP Kafka : 172.23.0.5
Connexion Kafka réussie !


In [4]:
from pyspark.sql.functions import to_json, struct

df_output = df_transformed.select(
    to_json(
        struct(
            "event_time",
            "city",
            "country",
            "temperature",
            "windspeed",
            "weather_code",
            "wind_alert_level",
            "heat_alert_level"
        )
    ).alias("value")
)

print("Données prêtes pour Kafka !")
df_output.printSchema()

Données prêtes pour Kafka !
root
 |-- value: string (nullable = true)



In [5]:
query = (
    df_output.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/tmp/kafka_weather_checkpoint")
    .outputMode("append")
    .start()
)

print("Streaming météo démarré !")

16:19:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming météo démarré !


16:19:38 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
16:19:40 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:19:40 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:19:40 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:19:40 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:19:40 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be establish

In [9]:
import time

time.sleep(10)

print("Streaming actif :", query.isActive)

print("Dernière progression :", query.lastProgress)

if query.exception() is not None:
    print("Erreur :", query.exception())

15:42:06 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:07 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:08 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:09 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:10 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:11 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not 

Streaming actif : True
Dernière progression : None


15:42:15 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:17 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:18 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:19 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
15:42:21 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not 

In [6]:
import socket

with socket.create_connection(("kafka", 9094), timeout=5):
    print("Kafka fonctionne avec Spark !")

Kafka fonctionne avec Spark !


16:19:58 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:19:59 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:00 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:01 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:03 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:04 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not 

In [7]:
# Lecture des données météo depuis Kafka

df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9094")
    .option("subscribe", "weather_stream")
    .option("startingOffsets", "earliest")
    .load()
)

print("Lecture Kafka configurée !")
df_kafka.printSchema()

Lecture Kafka configurée !
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



16:20:06 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:07 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:08 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:09 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.


In [8]:
from pyspark.sql.functions import (
    col, from_json, when
)

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)

# Définition du schéma des données Open-Meteo
schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", StructType([
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType())
    ]))
])

# Conversion des messages Kafka JSON
df_weather = (
    df_kafka
    .select(
        from_json(
            col("value").cast("string"),
            schema
        ).alias("data"),
        col("timestamp").alias("event_time")
    )
    .select("data.*", "event_time")
)

# Transformation des données et création des alertes
df_transformed = (
    df_weather
    .select(
        col("event_time"),
        col("city"),
        col("country"),
        col("current.temperature_2m").alias("temperature"),
        col("current.wind_speed_10m").alias("windspeed"),
        col("current.weather_code").alias("weather_code")
    )
    .withColumn(
        "wind_alert_level",
        when(col("windspeed") < 10, "level_0")
        .when(col("windspeed") <= 20, "level_1")
        .otherwise("level_2")
    )
    .withColumn(
        "heat_alert_level",
        when(col("temperature") < 25, "level_0")
        .when(col("temperature") <= 35, "level_1")
        .otherwise("level_2")
    )
    .filter(
        col("city").isNotNull()
        & col("temperature").isNotNull()
        & col("windspeed").isNotNull()
    )
)

print("Transformation météo configurée !")
df_transformed.printSchema()

Transformation météo configurée !
root
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- wind_alert_level: string (nullable = false)
 |-- heat_alert_level: string (nullable = false)



16:20:10 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:11 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:12 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:13 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:14 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:15 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not 

In [20]:
from pyspark.sql.functions import to_json, struct

df_output = df_transformed.select(
    to_json(
        struct(
            "event_time",
            "city",
            "country",
            "temperature",
            "windspeed",
            "weather_code",
            "wind_alert_level",
            "heat_alert_level"
        )
    ).alias("value")
)

print("Données prêtes pour Kafka !")
df_output.printSchema()

Données prêtes pour Kafka !
root
 |-- value: string (nullable = true)



In [9]:
# Arrêter une ancienne requête si elle fonctionne encore
if "query" in globals() and query.isActive:
    query.stop()

# Démarrage du streaming
query = (
    df_output.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9094")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/tmp/kafka_weather_checkpoint_v2")
    .outputMode("append")
    .start()
)

print("Streaming démarré !")
print("Streaming actif :", query.isActive)

16:20:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:17 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:18 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:19 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:20 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
16:20:21 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not 

Streaming démarré !
Streaming actif : True


16:20:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [10]:
import time

time.sleep(10)

print("Streaming actif :", query.isActive)
print("Dernière progression :", query.lastProgress)

if query.exception() is not None:
    print("Erreur :", query.exception())

Streaming actif : True
Dernière progression : None


16:23:46 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (172.23.0.3 executor 0): org.apache.kafka.common.errors.TimeoutException: Timeout of 60000ms expired before the position for partition weather_stream-0 could be determined

16:26:46 WARN TaskSetManager: Lost task 0.1 in stage 0.0 (TID 1) (172.23.0.3 executor 0): org.apache.kafka.common.errors.TimeoutException: Timeout of 60000ms expired before the position for partition weather_stream-0 could be determined

16:29:46 WARN TaskSetManager: Lost task 0.2 in stage 0.0 (TID 2) (172.23.0.3 executor 0): org.apache.kafka.common.errors.TimeoutException: Timeout of 60000ms expired before the position for partition weather_stream-0 could be determined

16:32:46 WARN TaskSetManager: Lost task 0.3 in stage 0.0 (TID 3) (172.23.0.3 executor 0): org.apache.kafka.common.errors.TimeoutException: Timeout of 60000ms expired before the position for partition weather_stream-0 could be determined

16:32:46 ERROR TaskSetManager: Task 0 in stage 0